- verl 的 FSDP worker 默认把训练侧 attention 实现设成了 flash_attention_2

```python
attn_implementation = self.override_config.get("attn_implementation", "flash_attention_2")
self.hf_config = AutoConfig.from_pretrained(
    self.local_hf_config_path, trust_remote_code=self.trust_remote_code, attn_implementation=attn_implementation
)
```

### sandbox env

- 沙盒环境：https://bytedance.github.io/SandboxFusion/
    - https://bytedance.github.io/SandboxFusion/docs/docs/get-started/#local-deployment
        - 部署与测试：可独立部署
        - `docker run -it -p 8080:8080 volcengine/sandbox-fusion:server-20250609`
            - a sandbox fusion server is needed to run the code interpreter tool.
    - endpoint 及返回结果
        - run_code: code interpreter
            - compile_result
            - run_result
        - get_prompts: dataset
        - submit:

- 部署：`docker run -it -p 8080:8080 volcengine/sandbox-fusion:server-20250609`
- 测试：
```sh
curl 'http://localhost:8080/run_code' \
  -H 'Content-Type: application/json' \
  --data-raw '{"code": "print(\"Hello, world!\")", "language": "python"}' | jq

{
  "status": "Success",
  "message": "",
  "compile_result": null,
  "run_result": {
    "status": "Finished",
    "execution_time": 0.07652020454406738,
    "return_code": 0,
    "stdout": "Hello, world!\n",
    "stderr": ""
  },
  "executor_pod_name": null,
  "files": {}
}
```

```
# server 端
INFO:     Application startup complete.
INFO:     Uvicorn running on http://:8080 (Press CTRL+C to quit)
2026-02-24 10:19:21 [debug    ] start processing python request with code ```
print("Hello, world!")
``` and files []... [sandbox.server.sandbox_api]
```

- docker stats
    - rollout 时，监控资源调用，主要吃 CPU，主要是科学计算，不吃 memory，不吃 GPU（没有训练任务）

#### 并发控制

- 模型动作调用 -> Ray tool actor -> Docker sandbox 服务
- num_workers: 128 / rate_limit: 128 (tool-config)
    - num_workers=128 不是创建 128 个 Docker，也不是创建 128 个 Python worker；它在 Ray 里创建一个名为 sandbox-execution-pool 的 actor，并把该 actor 的 max_concurrency 设为 128;
        - sandbox_fusion_tools.py:init_execution_pool
    - 每次模型发起 code_interpreter，会走：
        - ToolAgentLoop -> CustomSandboxFusionTool.execute() -> ExecutionWorker.execute.remote(...) -> requests.post(SANDBOX_FUSION_URL) -> Docker 内的 /run_code
    - rate_limit=128 是第二道限制：ExecutionWorker 里先从 TokenBucketWorker acquire 一个令牌，再真正执行 HTTP 请求：
- 同一个 Ray 集群里，最多大约 128 个 sandbox HTTP 请求同时飞向 http://10.1.123.37:8080/run_code。
- 并发的控制
    - `TokenBucketWorker`: `threading.Semaphore(rate_limit)`

| 层 | 参数 | 当前值 | 作用域 | 对并发的意义 |
|---|---|---:|---|---|
| AgentLoop 分片 | actor_rollout_ref.rollout.agent.num_workers | 默认 8 | AgentLoop worker 数 | 把轨迹切块并发跑 |
| 单轨迹工具并发 | actor_rollout_ref.rollout.multi_turn.max_parallel_calls | 默认 1 | 同一轮内 tool call | 每条轨迹每轮最多并行几个工具 |
| 工具执行池 | num_workers | 128 | 每个 Tool 实例自己的 ExecutionWorker actor | 该执行池的 max_concurrency |
| 全局限流 | enable_global_rate_limit=true, rate_limit=128 | 128 | 全局 TokenBucket（命名单例 actor） | 集群内同时执行的 tool 请求硬上限 |

### verl 中的 tool 定义与实现

> retool.py
- 定义 CustomSandboxFusionTool：把模型生成的代码清洗一下，再交给父类的 sandbox 执行逻辑。
    - CustomSandboxFusionTool 继承 verl 的 SandboxFusionTool，配置里把 code_interpreter 映射到 http://...:8080/run_code，训练 rollout 中的工具调用会经 Ray actor 池异步转发到 sandbox-fusion。
- 定义 CustomRLHFDataset：把数学数据集转成 tool_agent 可用的 prompt。
- 定义 compute_score：按最终答案打分，并略微鼓励多轮工具调用。

- retool.py: `class CustomSandboxFusionTool(SandboxFusionTool):`
    - `class SandboxFusionTool(BaseTool):`
        - create
        - execute
            - `self.execution_pool.execute.remote(self.execute_code, instance_id, code, timeout, language)`
    - CustomSandboxFusionTool 主要是重写了 execute 方法
        - 提取 Markdown 代码块 (预处理 1)
        - 强制打印最后一行结果 (预处理 2)
            - 因为后端的沙箱执行服务依赖 stdout（标准输出）来获取结果，如果大模型生成的代码只做了计算而没有 print，沙箱将返回空结果。所以这里它逆序遍历代码行，找到最后一行非空行，如果该行不是以 print 开头，就强制给它套上 print(...)。
    - CustomRLHFDataset
    - compute_score

```
model tool_call(code_interpreter)
    -> CustomSandboxFusionTool.execute()
    -> self.execution_pool.execute.remote(...)
    -> ExecutionWorker.execute(...)
    -> TokenBucketWorker.acquire()
    -> SandboxFusionTool.execute_code()
    -> _process_single_case()
    -> call_sandbox_api()
    -> POST sandbox_fusion_url /run_code
    -> ToolResponse(stdout + stderr)
```

#### ray actor （RPC）

```python
execution_pool = ray.remote(ExecutionWorker)
    .options(max_concurrency=num_workers)
    .remote(enable_global_rate_limit=enable_global_rate_limit, rate_limit=rate_limit)
```
- `ray.remote(ExecutionWorker)`: 把普通 Python 类 ExecutionWorker 包装成 Ray actor 类。
- `.options(max_concurrency=num_workers)`: 设置这个 actor 最多同时跑多少个方法调用。默认 Ray actor 通常一次只处理一个方法调用；这里设成 128 后，`ExecutionWorker.execute(...)` 最多可以有 128 个并发调用同时在 actor 内执行。
    - num_workers 不是“创建 128 个 actor”，而是：
    - one ExecutionWorker actor, up to num_workers concurrent method calls
- `.remote(...)`: 真正创建一个远程 actor 实例，并把 enable_global_rate_limit、rate_limit 传给 `ExecutionWorker.__init__。`
- `self.execution_pool` 不是一个本地对象，而是一个 Ray actor handle (`ActorHandle[ExecutionWorker]
`)。后面每次：
    - `await self.execution_pool.execute.remote(...)`
    - 都是往这个远程 actor 提交一个 execute 任务。
- `actual sandbox HTTP concurrency <= min(num_workers, rate_limit)`

```
AgentLoopWorker num_workers
  -> 有多少个 Ray actor 分摊 rollout batch

SandboxFusionTool num_workers
  -> 每个 sandbox 执行 actor 最多同时处理多少个 execute 调用

rate_limit
  -> 最多多少个调用真正进入 /run_code HTTP 请求
```

```python
@ray.remote(concurrency_groups={"acquire": 1, "release": 10})
class TokenBucketWorker:
    def __init__(self, rate_limit: int):
        self.rate_limit = rate_limit
        self.current_count = 0
        self._semaphore = threading.Semaphore(rate_limit)

    @ray.method(concurrency_group="acquire")
    def acquire(self):
        self._semaphore.acquire()
        self.current_count += 1

    @ray.method(concurrency_group="release")
    def release(self):
        self._semaphore.release()
        self.current_count -= 1

    def get_current_count(self):
        return self.current_count


class ExecutionWorker:
    def __init__(self, enable_global_rate_limit=True, rate_limit=10):
        self.rate_limit_worker = self._init_rate_limit(rate_limit) if enable_global_rate_limit else None

    def _init_rate_limit(self, rate_limit):
        return TokenBucketWorker.options(name="rate-limiter", get_if_exists=True).remote(rate_limit)

    def ping(self):
        return True

    def execute(self, fn: Callable[..., T], *fn_args, **fn_kwargs) -> T:
        if self.rate_limit_worker is None:
            return fn(*fn_args, **fn_kwargs)

        with ExitStack() as stack:
            stack.callback(self.rate_limit_worker.release.remote)
            ray.get(self.rate_limit_worker.acquire.remote())
            return fn(*fn_args, **fn_kwargs)


def init_execution_pool(
    num_workers: int, enable_global_rate_limit=True, rate_limit=10, mode: PoolMode = PoolMode.ThreadMode
):
    if mode == PoolMode.ThreadMode:
        return (
            ray.remote(ExecutionWorker)
            .options(max_concurrency=num_workers)
            .remote(enable_global_rate_limit=enable_global_rate_limit, rate_limit=rate_limit)
        )
    raise NotImplementedError("Process mode is not implemented yet")


def call_sandbox_api(
    sandbox_fusion_url: str,
    code: str,
    stdin: Optional[str],
    compile_timeout: int,
    run_timeout: int,
    memory_limit_mb: int,
    language: str = "python",
) -> tuple[Optional[dict[str, Any]], Optional[str]]:
    payload = {
        "compile_timeout": compile_timeout,
        "run_timeout": run_timeout,
        "code": code,
        "stdin": stdin,
        "memory_limit_MB": memory_limit_mb,
        "language": language,
        "files": {},
        "fetch_files": [],
    }

    try:
        response = requests.post(
            sandbox_fusion_url,
            headers={"Content-Type": "application/json", "Accept": "application/json"},
            data=json.dumps(payload),
            timeout=compile_timeout + run_timeout + 10,
        )
        response.raise_for_status()
        return response.json(), None
    ...


def _process_single_case(
    case_index: int,
    stdin_data: Any,
    expected_output: Any,
    sandbox_fusion_url: str,
    generation: str,
    timeout: int,
    memory_limit_mb: int,
    language: str,
) -> tuple[int, dict[str, Any]]:
    """Minimal version of the helper used by SandboxFusionTool.execute_code."""
    del case_index, expected_output
    api_response, error_msg = call_sandbox_api(
        sandbox_fusion_url=sandbox_fusion_url,
        code=generation,
        stdin=None if stdin_data is None else str(stdin_data),
        compile_timeout=timeout,
        run_timeout=timeout,
        memory_limit_mb=memory_limit_mb,
        language=language,
    )

    metadata = {
        "api_request_error": error_msg,
        "api_response": api_response,
        "run_status": None,
        "stdout": "",
        "stderr": "",
    }
    if error_msg or not api_response:
        return -1, metadata

    run_result = api_response.get("run_result") or {}
    metadata["run_status"] = run_result.get("status")
    metadata["stdout"] = run_result.get("stdout") or ""
    metadata["stderr"] = run_result.get("stderr") or ""
    return True, metadata


class SandboxFusionTool(BaseTool):
    """Compressed version of verl.tools.sandbox_fusion_tools.SandboxFusionTool."""

    def __init__(self, config: dict, tool_schema: OpenAIFunctionToolSchema):
        super().__init__(config, tool_schema)
        self._instance_dict = {}
        self.num_workers = config.get("num_workers", 10)
        self.rate_limit = config.get("rate_limit", 10)
        self.default_timeout = config.get("default_timeout", 30)
        self.default_language = config.get("default_language", "python")
        self.enable_global_rate_limit = config.get("enable_global_rate_limit", True)
        self.execution_pool = init_execution_pool(
            num_workers=self.num_workers,
            enable_global_rate_limit=self.enable_global_rate_limit,
            rate_limit=self.rate_limit,
            mode=PoolMode.ThreadMode,
        )
        self.sandbox_fusion_url = config.get("sandbox_fusion_url", "")
        self.memory_limit_mb = config.get("memory_limit_mb", 1024)
        if self.sandbox_fusion_url == "":
            raise ValueError("sandbox_fusion_url is not set")

    def get_openai_tool_schema(self) -> OpenAIFunctionToolSchema:
        return self.tool_schema

    async def create(
        self, instance_id: Optional[str] = None, ground_truth: Optional[str] = None, **kwargs
    ) -> tuple[str, ToolResponse]:
        if instance_id is None:
            instance_id = str(uuid4())
        self._instance_dict[instance_id] = {
            "response": "",
            "ground_truth": ground_truth,
            "reward": [],
        }
        return instance_id, ToolResponse()

    async def execute(self, instance_id: str, parameters: dict[str, Any], **kwargs) -> tuple[ToolResponse, None, None]:
        code = parameters.get("code", "")
        timeout = parameters.get("timeout", self.default_timeout)
        language = parameters.get("language", self.default_language)
        if not isinstance(code, str):
            code = str(code)

        result = await self.execution_pool.execute.remote(self.execute_code, instance_id, code, timeout, language)
        return result, None, None

    def execute_code(self, instance_id, code, timeout=30, language="python"):
        result_status, metadata = _process_single_case(
            0, None, None, self.sandbox_fusion_url, code, timeout, self.memory_limit_mb, language
        )
        del result_status
        if metadata["run_status"] == "Finished":
            actual_output = metadata["stdout"] + metadata["stderr"]
            return ToolResponse(text=actual_output)
        return ToolResponse(text="no stdout here")

    async def calc_reward(self, instance_id: str, **kwargs) -> str:
        return self._instance_dict[instance_id]["reward"]

    async def release(self, instance_id: str, **kwargs) -> None:
        del self._instance_dict[instance_id]


class CustomSandboxFusionTool(SandboxFusionTool):
    """Compressed version of recipe.retool.retool.CustomSandboxFusionTool."""

    code_pattern = re.compile(r"```python(.*?)```", re.DOTALL)

    async def execute(self, instance_id: str, parameters: dict[str, Any], **kwargs) -> tuple[ToolResponse, None, None]:
        code = parameters["code"]
        matches = self.code_pattern.findall(code)
        if matches:
            code = matches[0].strip()

        # Match ReTool's behavior: print the last non-empty expression if needed.
        lines = code.split("\n")
        for i, line in reversed(list(enumerate(lines))):
            if line == "":
                continue
            if not lines[i].startswith("print"):
                lines[i] = f"print({line})"
            break
        code = "\n".join(lines)

        timeout = parameters.get("timeout", self.default_timeout)
        language = parameters.get("language", self.default_language)
        if not isinstance(code, str):
            code = str(code)

        result = await self.execution_pool.execute.remote(self.execute_code, instance_id, code, timeout, language)
        return result, None, None
```

```python
async def one_rollout_turn() -> None:
    if not ray.is_initialized():
        ray.init(ignore_reinit_error=True)

    tool_schema = OpenAIFunctionToolSchema(function={"name": "code_interpreter"})
    tool = CustomSandboxFusionTool(
        config={
            "sandbox_fusion_url": "http://localhost:8080/run_code",
            "num_workers": 128,
            "enable_global_rate_limit": True,
            "rate_limit": 128,
            "default_timeout": 30,
            "default_language": "python",
            "memory_limit_mb": 1024,
            "type": "native",
        },
        tool_schema=tool_schema,
    )

    # This is the parsed form of a model-emitted code_interpreter tool call.
    tool_call_arguments = {
        "code": """
```python
x = 25 / 11
(1 / 3) * (x + 1 / x - 1)
\```
""",
    }

    instance_id, _ = await tool.create(instance_id="one-rollout-trajectory")
    try:
        tool_response, _, _ = await tool.execute(instance_id=instance_id, parameters=tool_call_arguments)
    finally:
        await tool.release(instance_id)

    # In verl, this text becomes a role="tool" message and is appended to the
    # prompt before the next assistant generation.
    print("tool message:", tool_response.text)
```